# ДЗ-4. Рекомендательные системы и Spark MLlib

**Фамилия:** Николаев  
**Вариант для задачи 1:** 1 — Animation, Romance, Documentary  
**Вариант для задачи 2:** 1 — коллаборативная фильтрация по схожести пользователей (user-based)

Датасет: MovieLens, небольшой набор `ml-latest-small` (≈100k рейтингов).

Проверка варианта:
```python
alp = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
w = [1,42,21,21,34,6,44,26,18,44,38,26,14,43,4,49,45,7,42,29,4,9,36,34,31,29,5,30,4,19,28,25,33]
d = dict(zip(alp, w))
variant = sum(d[el] for el in 'николаев') % 40 + 1   # = 6
task1 = variant % 3 + 1   # = 1
task2 = variant % 2 + 1   # = 1
```

## 0. Инициализация Spark

In [ ]:
import os, sys, math
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import DoubleType

spark = (
    SparkSession.builder
        .appName('BigData_HW4')
        .master('local[*]')
        .config('spark.sql.shuffle.partitions', '8')
        .config('spark.driver.memory', '2g')
        .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
spark

In [ ]:
DATA_DIR = '../data/ml-latest-small'
GENRES = ['Animation', 'Romance', 'Documentary']
SEED = 42

ratings = spark.read.csv(f'{DATA_DIR}/ratings.csv', header=True, inferSchema=True)
movies  = spark.read.csv(f'{DATA_DIR}/movies.csv',  header=True, inferSchema=True)
movies_exploded = movies.withColumn('genre', F.explode(F.split(F.col('genres'), '\\|')))

print('ratings:', ratings.count(), '| movies:', movies.count())
ratings.show(3); movies.show(3, truncate=False)

## Задание 1. Анализ датасета

Жанры варианта 1: **Animation, Romance, Documentary**.  
Один фильм может относиться к нескольким жанрам, поэтому строку `genres` мы предварительно разнесли по строкам через `explode(split(...))`.

### 1.1 Количество фильмов по всем жанрам

In [ ]:
(movies_exploded
    .groupBy('genre')
    .agg(F.countDistinct('movieId').alias('movies_cnt'))
    .orderBy(F.desc('movies_cnt'))
    .show(50, truncate=False))

### 1.2–1.5 Топы фильмов по жанрам варианта

Считаем количество и среднее значение рейтингов на фильм, присоединяем к (фильм × жанр) и фильтруем по жанрам варианта.

In [ ]:
movie_stats = (
    ratings.groupBy('movieId').agg(
        F.count('rating').alias('ratings_cnt'),
        F.avg('rating').alias('avg_rating'),
    )
)
movie_full = (
    movies_exploded
        .join(movie_stats, on='movieId', how='left')
        .fillna({'ratings_cnt': 0, 'avg_rating': 0.0})
)
movie_full.show(5, truncate=False)

In [ ]:
def show_genre_tops(genre: str, n: int = 10):
    g = movie_full.filter(F.col('genre') == genre)
    cols = ['movieId', 'title', 'ratings_cnt', 'avg_rating']

    print(f'\n=== {genre}: топ-{n} по числу рейтингов ===')
    g.orderBy(F.desc('ratings_cnt'), 'title').select(cols).show(n, truncate=False)

    g10 = g.filter(F.col('ratings_cnt') > 10)
    print(f'=== {genre}: топ-{n} по наименьшему числу рейтингов (>10) ===')
    g10.orderBy(F.asc('ratings_cnt'), 'title').select(cols).show(n, truncate=False)

    print(f'=== {genre}: топ-{n} по наибольшему среднему рейтингу (#ratings>10) ===')
    g10.orderBy(F.desc('avg_rating'), F.desc('ratings_cnt')).select(cols).show(n, truncate=False)

    print(f'=== {genre}: топ-{n} по наименьшему среднему рейтингу (#ratings>10) ===')
    g10.orderBy(F.asc('avg_rating'), F.desc('ratings_cnt')).select(cols).show(n, truncate=False)

for g in GENRES:
    show_genre_tops(g)

## Задание 2. Коллаборативная фильтрация (user-based)

**План:**
1. Делим рейтинги на `train_init` (0.8) и `test` (0.2). Считаем средний рейтинг в `train_init` и RMSE на `test` для предсказания «всем — средний» (это baseline).
2. Считаем схожесть пользователей по центрированным рейтингам (косинус по центрированным = Pearson). Берём топ-K соседей. Прогноз: средний рейтинг пользователя плюс взвешенное среднее отклонений соседей по этому фильму. Где соседей нет — fallback на средний пользователя или общий средний.
3. Считаем RMSE на `test`.

In [ ]:
train_init, test = ratings.randomSplit([0.8, 0.2], seed=SEED)
train_init.cache(); test.cache()
print('train_init:', train_init.count(), '| test:', test.count())

mean_rating = train_init.agg(F.avg('rating')).first()[0]
print(f'Mean rating (train): {mean_rating:.4f}')

rmse_baseline = math.sqrt(
    test.select(((F.col('rating') - F.lit(mean_rating)) ** 2).alias('se'))
        .agg(F.avg('se')).first()[0]
)
print(f'RMSE baseline (mean): {rmse_baseline:.4f}')

In [ ]:
# 1) центрируем рейтинги пользователя
user_means = train_init.groupBy('userId').agg(F.avg('rating').alias('user_mean'))
train_centered = (
    train_init.join(user_means, on='userId')
              .withColumn('r_centered', F.col('rating') - F.col('user_mean'))
)

# 2) пары пользователей через общий фильм
a = train_centered.alias('a')
b = train_centered.alias('b')
pairs = (
    a.join(b, on='movieId')
     .filter(F.col('a.userId') < F.col('b.userId'))
     .select(
         F.col('a.userId').alias('u'),
         F.col('b.userId').alias('v'),
         F.col('a.r_centered').alias('ru'),
         F.col('b.r_centered').alias('rv'),
     )
)

sim = (
    pairs.groupBy('u', 'v').agg(
        F.sum(F.col('ru') * F.col('rv')).alias('dot'),
        F.sum(F.col('ru') * F.col('ru')).alias('nu'),
        F.sum(F.col('rv') * F.col('rv')).alias('nv'),
        F.count('*').alias('common'),
    )
    .filter((F.col('common') >= 3) & (F.col('nu') > 0) & (F.col('nv') > 0))
    .withColumn('sim', F.col('dot') / (F.sqrt('nu') * F.sqrt('nv')))
    .select('u', 'v', 'sim')
)

# симметризуем
sim_full = sim.union(sim.select(F.col('v').alias('u'), F.col('u').alias('v'), 'sim'))

# 3) топ-K соседей у каждого
K = 30
w = Window.partitionBy('u').orderBy(F.desc('sim'))
top_neighbors = (
    sim_full.withColumn('rk', F.row_number().over(w))
            .filter(F.col('rk') <= K)
            .select(F.col('u').alias('userId'), F.col('v').alias('neighborId'), 'sim')
)
top_neighbors.show(5)

In [ ]:
# 4) предсказания на test
predictions = (
    test.alias('t')
        .join(top_neighbors.alias('n'), on='userId', how='left')
        .join(
            train_centered.select(
                F.col('userId').alias('neighborId'),
                F.col('movieId'),
                F.col('r_centered'),
            ).alias('nc'),
            on=['neighborId', 'movieId'], how='left'
        )
        .filter(F.col('r_centered').isNotNull())
        .groupBy('userId', 'movieId')
        .agg(
            F.sum(F.col('sim') * F.col('r_centered')).alias('num'),
            F.sum(F.abs(F.col('sim'))).alias('den'),
        )
        .filter(F.col('den') > 0)
        .withColumn('pred_offset', F.col('num') / F.col('den'))
)

predictions = (
    predictions.join(user_means, on='userId', how='left')
               .withColumn('prediction',
                           F.coalesce('user_mean', F.lit(mean_rating)) + F.col('pred_offset'))
               .withColumn('prediction', F.greatest(F.lit(0.5), F.col('prediction')))
               .withColumn('prediction', F.least(F.lit(5.0), F.col('prediction')))
               .select('userId', 'movieId', 'prediction')
)

test_with_pred = (
    test.join(predictions, on=['userId', 'movieId'], how='left')
        .join(user_means, on='userId', how='left')
        .withColumn('prediction', F.coalesce('prediction', 'user_mean', F.lit(mean_rating)))
)
test_with_pred.select('userId', 'movieId', 'rating', 'prediction').show(10)

In [ ]:
se = ((F.col('rating') - F.col('prediction')) ** 2).cast(DoubleType())
rmse_cf = math.sqrt(test_with_pred.select(F.avg(se).alias('mse')).first()[0])

print(f'RMSE baseline (mean): {rmse_baseline:.4f}')
print(f'RMSE user-based CF:   {rmse_cf:.4f}')

In [ ]:
spark.stop()